# 그리디 알고리즘(Greedy algorithm)

------------------------------

**(코랩에서)한글 폰트 지정하는 방법**

In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 코랩에서 위 코드를 실행시킨 후  반드시 코랩 메뉴: "런타임>세션 다시 시작" 합니다.

In [ ]:
# korean font
# Colab: NanumGothic, Mac: AppleGothic, 윈도우: Malgun Gothic
fontname = 'NanumGothic'
figsize = (5, 3)
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': fontname,        # (코랩)한글 폰트
                     'font.size': 12,
                     'figure.figsize': figsize,
                     'axes.unicode_minus':  False }) # 폰트 설정





---



# 1.그리디 알고리즘(탐욕 알고리즘)

## 1-1.알고리즘 설계 전략



---



## 1-2.그리디 알고리즘

### 1) 그리디 알고리즘 소개
- 그리디 알고리즘(탐욕 알고리즘)은 최적화 문제를 해결하기 위해 고안된 간단하면서도 효율적인 방법론,
- 이 알고리즘의 핵심은 매 선택 시점에서 지역적으로 가장 좋은 것을 선택함으로써, 전체적인 해답을 구하는 방식
- "그리디(greedy)"는 '탐욕스러운'이라는 뜻으로, 매 순간 최적의 해를 탐욕스럽게 선택한다는 의미에서 붙여짐 --> 탐욕 알고리즘

### [예시] 감시 카메라 최소 설치 문제 - 그리디 알고리즘으로 근사해 구하기
1. 문제 정의 (구역, 카메라 커버 영역)
2. 그리디 알고리즘을 통한 근사해 계산
3. networkx를 활용한 알고리즘 전/후 시각화

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

def draw_coverage_graph(cameras, universe, covered, title, selected_cameras=None, figsize=figsize):
    """
    감시 카메라 커버리지 시각화 함수 (기본 or 강조 모드)

    Parameters:
    - cameras: dict, 각 카메라의 커버 영역
    - universe: set, 전체 구역
    - covered: set, 커버된 구역
    - title: str, 그래프 제목
    - selected_cameras: list, 강조할 카메라 목록 (없으면 기본 모드)
    - figsize: tuple, 그래프 크기
    """

    G = nx.Graph()

    # 노드 추가
    for cam in cameras:
        G.add_node(cam, bipartite=0)
    for zone in universe:
        G.add_node(zone, bipartite=1)

    edges = []
    edge_colors = []
    edge_widths = []

    # 엣지 및 스타일 설정
    for cam, zones in cameras.items():
        for zone in zones:
            G.add_edge(cam, zone)
            edges.append((cam, zone))
            if selected_cameras and cam in selected_cameras:
                edge_colors.append('blue')
                edge_widths.append(2.5)
            else:
                edge_colors.append('gray')
                edge_widths.append(1.0)

    # 노드 색상
    node_colors = []
    for node in G.nodes():
        if node in universe:
            node_colors.append('green' if node in covered else 'red')
        else:
            node_colors.append('skyblue')

    # 배치 및 시각화
    pos = nx.bipartite_layout(G, [cam for cam in cameras])
    plt.figure(figsize=figsize)
    nx.draw(G, pos, with_labels=True, node_color=node_colors,
            edge_color=edge_colors, width=edge_widths,
            node_size=800, font_size=10)
    plt.title(title)
    plt.show()


# Greedy Set Cover 알고리즘
def greedy_set_cover(universe, subsets):
    covered = set()
    selected_cameras = []

    while covered != universe:
        best_camera = None
        best_covered = set()

        for camera, zones in subsets.items():
            #--------------------------------
            # Greedy전략: 매 순간 가장 효율적으로 많은 구역을 덮는 카메라를 선택하는 방식
            uncovered = zones - covered
            if len(uncovered) > len(best_covered):
                best_camera = camera
                best_covered = uncovered
            #--------------------------------

        if best_camera is None:
            break

        selected_cameras.append(best_camera)
        covered |= subsets[best_camera]

        print(f"✅ 선택된 카메라: {best_camera}, 새로 커버한 영역: {best_covered}")

    return selected_cameras, covered


# ----- 실행 -----
# 감시 대상 전체 구역
universe = {'A', 'B', 'C', 'D', 'E', 'F', 'G'}

# 각 카메라가 커버할 수 있는 구역 (부분 집합)
cameras = {
    'Cam1': {'A', 'B', 'C'},
    'Cam2': {'C', 'D', 'E'},
    'Cam3': {'E', 'F'},
    'Cam4': {'G'},
    'Cam5': {'B', 'F', 'G'}
}
print(f'cameras:\n{cameras}\n')

# 1. 알고리즘 실행
selected_cameras, covered_zones = greedy_set_cover(universe, cameras)

# 2. 그래프 시각화 (전->후)
draw_coverage_graph(cameras, universe, set(), "초기 커버 상태")
draw_coverage_graph(cameras, universe, covered_zones, "Greedy 결과: 선택된 카메라 강조",
                    selected_cameras=selected_cameras)

# 3. 최종 선택된 카메라 확인
print("\n✅ 최종 선택된 카메라 목록:", selected_cameras)




---



## 1-3.그리디 알고리즘 작동 원리

1. 문제 분할
    - 문제를 여러 개의 하위 문제로 나눈다
    - 하위 문제는 특정 자원을 어떻게 배분하거나, 어떤 선택을 할지 결정하는 것과 관련이 있다.
2. 선택 절차
    - 각 단계에서 최적의 선택을 한다.
    - 현재 주어진 정보만을 바탕으로 이루어짐
3. 해결책 구성
    - 각 단계의 선택이 연속적으로 이루어지면서 최종적인 해결책이 구성된다.
4. 종료 조건
    - 모든 자원이 배분되거나, 모든 선택이 완료되면 알고리즘은 종료된다.

### 1) [예시1] : 동전 교환 문제(최소 개수)
- 문제 : 고객에게 거슬러 줘야 할 돈 N(1260)원이 있을 때, 최소한의 동전 개수로 거슬러 주는 방법은?
- 해결 : **가장 큰 동전부터 가능한 많이 사용**하여 금액을 채운다. 각 단계에서 가능한 최대 금액을 만드는 동전을 선택한다

In [ ]:
def greedy_coin_change(coins, amount):
    coins.sort(reverse=True)  # 가장 큰 동전부터 선택
    count = 0
    for coin in coins:
        while amount >= coin:
            amount -= coin
            count += 1
    return count

# 테스트 예시
coins = [500, 100, 50, 10]  # 동전의 종류
amount = 1260               # 거슬러 줘야 할 금액
count = greedy_coin_change(coins, amount)
print(f"✅ 총 동전 개수: {count}")

In [ ]:
# Coin Change (동전 교환) – Greedy 알고리즘
def greedy_coin_change(coins, amount):
    coins.sort(reverse=True)  # 가장 큰 동전부터 선택
    count = 0
    result = []
    for coin in coins:
        num = amount // coin   # 몫 -> 동전 개수
        amount %= coin         # 나머지 -> 다시 나눌 금액
        count += num           # 동전 개수
        result.append((coin, num))
    return count, result

# 테스트 예시
coins = [500, 100, 50, 10]  # 동전의 종류
amount = 1260               # 거슬러 줘야 할 금액
count, result = greedy_coin_change(coins, amount)
print(f"✅ 총 동전 개수: {count}")
print('-'*30)
for coin, num in result:
    print(f"{coin:>4}원: {num:>2}개")


### 2) [예시2]: 활동 선택 문제(최대 개수)
- 문제 : 시작/종료 시간이 주어진 활동 중 내가 선택할 수 있는 최대 활동 개수는?
- 해결책 : 활동이 겹치지 않도록 **종료 시간이 가장 빠른 활동을 먼저** 고려한다.

In [ ]:
# Activity Selection (활동 선택) – Greedy 알고리즘
def activity_selection(activities):
    # 종료 시간 기준 정렬 (Greedy 선택 기준)
    activities.sort(key=lambda x: x[1])

    selected = []    # 선택된 활동 리스트
    last_end = 0     # 직전에 선택된 활동의 종료시간: 0/-float("inf")

    for start, end in activities:
        if start >= last_end:    # 현재 시작시간이 직전 종료시간 보다 크거나 같으면
            selected.append((start, end))
            last_end = end

    return selected

# 활동 정의
activities = [(1, 3), (2, 5), (3, 9), (6, 8)]

# 최대 활동 선택 함수 호출
result = activity_selection(activities)

print(f"✅ 선택된 활동 개수: {len(result)}")
print("✅ 선택된 활동 목록 (시작, 종료):")
for s, e in result:
    print(f"  • ({s}, {e})")


### 3) [예시3]: 회의실 예약(최소 개수)
- 문제 : 여러 회의들의 시작시간과 종료시간이 주어졌을 때, 모든 회의가 이루어지도록 필요한 최소 회의실 수는?
- 해결 : 회의 시간이 겹치지 않게  **시작시간/종료시간 기준 정렬**

- **종료시간 기준**으로 + 리스트 사용 + 시각화

In [ ]:
# 종료시간 기준으로 정렬 -->  회의실 배정 --> 간트 차트로 시각화
import matplotlib.pyplot as plt

def visualize_schedule_gantt(meetings, room_assignments):
    """
    회의 배정을 간트 차트로 시각화
    :meetings: [(start, end)]
    :room_assignments: 각 회의가 배정된 회의실 번호 리스트
    """
    fig, ax = plt.subplots(figsize=figsize)
    colors = plt.get_cmap('tab20', len(set(room_assignments)))

    for i, ((start, end), room) in enumerate(zip(meetings, room_assignments)):
        ax.broken_barh([(start, end - start)], (room - 0.4, 0.8),
                       facecolors=colors(room),
                       edgecolor='black')
        ax.text(start + (end - start)/2, room, f"회의{i+1}", va='center', ha='center', color='white', fontsize=9)

    ax.set_yticks(sorted(set(room_assignments)))
    ax.set_yticklabels([f"회의실 {r}" for r in sorted(set(room_assignments))])
    ax.set_xlabel("시간")
    ax.set_title("최소 회의실 배정 현황")
    ax.grid(True)
    plt.tight_layout()
    plt.show()

#-----------------------------
# 회의실 배정 알고리즘 + 회의실 번호 추적
def meeting_rooms_assign(meetings):
    if not meetings:
        return 0

    meetings.sort(key=lambda x: x[1])       # 종료시간 기준 정렬
    end_times = []                          # 종료시간
    room_assignment = [0] * len(meetings)   # 미팅별 회의실 배정 리스트
    print(meetings)

    # 현재 회의를 기존 회의실 중 하나에 배정할 수 있는지 확인
    for idx, (start, end) in enumerate(meetings):
        assigned = False
        for room_idx in range(len(end_times)):
            if start >= end_times[room_idx]:    # 기존 회의실 배정 조건
                end_times[room_idx] = end
                room_assignment[idx] = room_idx + 1
                assigned = True
                break

        if not assigned:   # 새로운 회의실 배정
            end_times.append(end)
            room_assignment[idx] = len(end_times)

    print('room_assignment: ', room_assignment)
    print('end_times:', end_times)
    return room_assignment



# 회의 목록 (시작, 종료)
meetings = [
    (10.00, 12.00),
    (9.00, 10.30),
    (11.00, 13.00),
    (12.30, 14.00),
    (15.00, 16.30)
]

# 회의실 배정 실행
assignments = meeting_rooms_assign(meetings)
print("✅ 최소 회의실 수:", len(set(assignments)))

# 회의실 배정 현황 간트 차트로 시각화
visualize_schedule_gantt(meetings, assignments)


- **시작시간 기준**으로 + 리스트 사용

In [ ]:
# Meeting_rooms Assign (회의실 배정) – Greedy 알고리즘
def meeting_rooms_assign(meetings):
    if not meetings:
        return 0

    # 1. 시작시간 기준 정렬
    meetings.sort(key=lambda x: x[0])       # 시작시간 기준 정렬
    end_times = []                          # 종료시간
    room_assignment = [0] * len(meetings)   # 미팅별 회의실 배정 리스트
    print(meetings)


    # 전체 미팅에 대해서 반복하기
    for idx, (start, end) in enumerate(meetings):
        assigned = False

        # 현재 회의를 기존 회의실 중 하나에 배정할 수 있는지 확인
        for room_idx, e in enumerate(end_times):
            if start >= e:                      # 기존 회의실에 배정 조건
                end_times[room_idx] = end       # 해당 회의실 종료시간 업데이트
                room_assignment[idx] = room_idx + 1
                assigned = True
                break

        # 기존 회의실 중 배정할 수 있는 곳이 없으면 새 회의실 추가
        if not assigned:
            end_times.append(end)
            room_assignment[idx] = len(end_times)


    print('room_assignment: ', room_assignment)
    print('end_times:', end_times)
    return room_assignment


# 회의실 배정 실행
assignments = meeting_rooms_assign(meetings)
print("✅ 최소 회의실 수:", len(set(assignments)))

# 회의실 배정 현황 간트 차트로 시각화
visualize_schedule_gantt(meetings, assignments)


- **종료시간 기준**으로 + heapq 사용

In [ ]:
# 종료시간 기준으로 정렬 + heapq 사용
import heapq

def min_meeting_rooms(meetings):
    if not meetings:
        return 0

    # 입력된 회의 시간을 종료 시간 기준으로 정렬
    meetings.sort(key=lambda x: x[1])

    # 최소 힙을 사용하여, 현재 회의실에서 가장 빨리 끝나는 회의의 종료 시간을 관리
    rooms = []  # 종료시간
    heapq.heappush(rooms, meetings[0][1])

    for i in range(1, len(meetings)):
        # 가장 빨리 끝나는 회의의 종료시간보다 현재 회의의 시작시간이 크거나 같다면, 동일 회의실 사용 가능
        if meetings[i][0] >= rooms[0]:
            heapq.heappop(rooms)
        # 새 회의실 배정
        heapq.heappush(rooms, meetings[i][1])

    # 힙의 크기가 필요한 최소 회의실 수
    print("✅ rooms 종료시간: ",rooms)
    return len(rooms)



# 함수 호출 및 출력
print("✅ 필요한 최소 회의실 수:", min_meeting_rooms(meetings))

### [실습문제] 요일별 최소 회의실 예약
- 문제 : 요일, 시작시간, 종료시간이 주어졌을 때, 모든 회의가 이루어지도록 필요한 최소 회의실 수는?

In [ ]:
import heapq

# 앞에서 정의된 함수 재사용
# def min_meeting_rooms(meetings):

def meetings_by_day(meetings):
    # 요일별로 회의 분류
    weekly_meetings = {}
    for day, start, end in meetings:
        if day not in weekly_meetings:
            weekly_meetings[day] = []
        weekly_meetings[day].append((start, end))

    # 요일별로 필요한 최소 회의실 수 계산
    min_rooms_per_day = {}
    for day, intervals in weekly_meetings.items():
        min_rooms_per_day[day] = min_meeting_rooms(intervals)

    return min_rooms_per_day

# 예제 입력 데이터
meetings = [
    ("Monday", 9, 12), ("Monday", 11, 14),
    ("Tuesday", 9, 10), ("Tuesday", 10, 11), ("Tuesday", 11, 12),
    ("Wednesday", 14, 18), ("Wednesday", 15, 19), ("Wednesday", 19, 20)
]

# 함수 호출 및 출력
meeting_rooms_needed = meetings_by_day(meetings)
print("필요한 최소 회의실 수:", meeting_rooms_needed)

- 요일별 최소 회의실 수 & 할당된 회의실ID

In [ ]:
import heapq
from collections import defaultdict

# 하나의 요일에 대해 최소 회의실 수와 회의실 ID 배정
def assign_meeting_rooms(intervals):
    # 회의 시작 시간 기준 정렬
    intervals.sort()
    # 힙: (종료 시간, 회의실 ID)
    heap = []
    room_id_counter = 0
    assigned = []  # (start, end, room_id)

    for start, end in intervals:
        if heap and heap[0][0] <= start:
            # 기존 회의실 재사용
            earliest_end, room_id = heapq.heappop(heap)
        else:
            # 새로운 회의실 필요
            room_id = room_id_counter
            room_id_counter += 1

        heapq.heappush(heap, (end, room_id))
        assigned.append((start, end, room_id))

    return room_id_counter, assigned

# 요일별 회의실 배정 및 출력
def meetings_by_day_with_rooms(meetings):
    weekly_meetings = defaultdict(list)

    # 요일별 회의 분류
    for day, start, end in meetings:
        weekly_meetings[day].append((start, end))

    # 요일별 회의실 배정 결과 저장
    results = {}
    for day, intervals in weekly_meetings.items():
        num_rooms, assigned = assign_meeting_rooms(intervals)
        results[day] = {
            "min_rooms": num_rooms,
            "assignments": assigned
        }

    return results

# 예제 입력 데이터
meetings = [
    ("Monday", 9, 12), ("Monday", 11, 14),
    ("Tuesday", 9, 10), ("Tuesday", 10, 11), ("Tuesday", 11, 12),
    ("Wednesday", 14, 18), ("Wednesday", 15, 19), ("Wednesday", 19, 20)
]

# 결과 출력
meeting_rooms_info = meetings_by_day_with_rooms(meetings)
for day, info in meeting_rooms_info.items():
    print(f"\n📅 {day}:")
    print(f"   🔢 필요한 최소 회의실 수: {info['min_rooms']}")
    for start, end, room_id in sorted(info['assignments']):
        print(f"   🕒 {start}~{end}시 → 회의실 {room_id}")


In [ ]:
from collections import defaultdict

# 1. 회의실 배정 함수 (종료시간 기준 정렬, 기존 회의실 재사용 우선)
def meeting_rooms_assign(meetings):
    if not meetings:
        return []

    meetings.sort(key=lambda x: x[1])  # 종료시간 기준 정렬
    end_times = []  # 각 회의실의 마지막 종료시간
    room_assignment = [0] * len(meetings)

    for idx, (start, end) in enumerate(meetings):
        assigned = False
        for room_idx in range(len(end_times)):
            if start >= end_times[room_idx]:
                end_times[room_idx] = end
                room_assignment[idx] = room_idx + 1
                assigned = True
                break
        if not assigned:
            end_times.append(end)
            room_assignment[idx] = len(end_times)

    return room_assignment

# 2. 요일별로 회의 분류 및 회의실 배정
def meetings_by_day_assign(meetings):
    weekly_meetings = defaultdict(list)
    for day, start, end in meetings:
        weekly_meetings[day].append((start, end))

    results = {}
    for i, (day, intervals) in enumerate(weekly_meetings.items()):
        assignments = meeting_rooms_assign(intervals)
        results[day] = {
            "intervals": intervals,
            "assignments": assignments,
            "day_index": i
        }

        # 요일별 회의실 배정 정보 출력
        print(f"\n📅 {day}:")
        print(f"   ✅ 필요한 최소 회의실 수: {len(set(assignments))}")
        for idx, ((start, end), room_id) in enumerate(zip(intervals, assignments)):
            print(f"   회의 {idx+1}: {start}~{end}시 → 회의실 {room_id}")

    return results

# 3. 예제 입력 데이터
meetings = [
    ("Monday", 9, 12), ("Monday", 11, 14),
    ("Tuesday", 9, 10), ("Tuesday", 10, 11), ("Tuesday", 11, 12),
    ("Wednesday", 14, 18), ("Wednesday", 15, 19), ("Wednesday", 19, 20)
]

# 4. 실행
weekly_assignments = meetings_by_day_assign(meetings)


----------------------------

## 1-4.응용 :  허프만 코딩(Huffman coding)

### 1) 허프만 코딩 소개
- 허프만 코딩은 **문자 빈도에 따라 가변 길이의 비트 코드를 할당**하여 **데이터를 효율적으로 압축**하는 방법
- 이 방법은 **자주 사용되는 문자에 더 짧은 코드를, 덜 사용되는 문자에 더 긴 코드를 할당**함으로써 전체 메시지의 비트 수를 최소화함


### 2) 허프만 코딩 알고리즘의 작동 원리
- **빈도 수 계산**: 주어진 데이터에서 각 문자의 빈도 수를 계산합니다.
- **우선순위 큐 생성**: 각 문자를 빈도 수에 따라 우선순위 큐(최소 힙)에 삽입합니다.
- **트리 생성**: 가장 빈도 수가 낮은 두 노드를 큐에서 꺼내 이들을 부모 노드로 하는 새 노드를 생성하고, 이 새 노드를 큐에 다시 삽입합니다. 이 과정을 큐에 하나의 노드만 남을 때까지 반복합니다. 이 하나의 노드는 전체 문자의 허프만 트리의 루트 노드가 됩니다.
- **코드 할당**: 루트 노드에서 시작하여 각 리프 노드(문자)까지의 경로를 따라 코드를 할당합니다. 왼쪽 자식에게는 '0', 오른쪽 자식에게는 '1'을 할당합니다.

- **[예시] 텍스트 파일 압축**

In [ ]:
import heapq
from collections import Counter, defaultdict

class Node:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None

    def __lt__(self, other):
        return self.freq < other.freq

    def __repr__(self):
        return f"{self.char}:{self.freq}"


# 허프만 트리에서 코드 할당
def encode(node, prefix="", code_map={}):
    if node is None:
        return
    if node.char is not None:  # 리프 노드인 경우
        code_map[node.char] = prefix
    encode(node.left, prefix + "0", code_map)
    encode(node.right, prefix + "1", code_map)
    return code_map

# 허프만 트리 생성
def huffman_tree(data):
    # 문자 빈도 수 계산 및 힙 생성
    freq = Counter(data)
    heap = []
    for char, frequency in freq.items():
        heapq.heappush(heap, Node(char, frequency))
    print('heap:', heap)  # 최소힙? or 최대힙?

    # 힙을 사용하여 허프만 트리 생성
    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        merged = Node(None, left.freq + right.freq)
        merged.left = left
        merged.right = right
        heapq.heappush(heap, merged)

    root = heapq.heappop(heap) # 트리의 루트 노드를 얻음
    code_map = encode(root)    # 허프만 트리에서 코드 할당
    encoded_output = "".join(code_map[char] for char in data)  # 압축된 데이터 생성

    return root, code_map, encoded_output


data = "this is an example of a huffman tree"
root, code_map, encoded_output  = huffman_tree(data) # 허프만 트리 생성

# 출력
print("\n✅ 입력 문자열:", data)
print("\n✅ 허프만 코드맵:")
for char, code in code_map.items():
    print(f"  '{char}': {code}")

print("\n✅ 압축 결과:")
original_bits = len(data) * 8
compressed_bits = len(encoded_output)
compression_ratio = compressed_bits / original_bits

print(f"  - 원본 비트 수:     {original_bits} bits")
print(f"  - 압축 후 비트 수:  {compressed_bits} bits")
print(f"  - 압축 비율:        {compression_ratio:.2%}")
print(f"\nEncoded Output:\n{encoded_output}")


- **heapq_tree 시각화**

In [ ]:
# heapq_tree 시각화
import networkx as nx
import matplotlib.pyplot as plt

def add_nodes_edges(node, graph, pos, x=0, y=0, layer=1):
    pos[node] = (x, y)
    if node.left:
        graph.add_edge(node, node.left, weight='0')
        add_nodes_edges(node.left, graph, pos, x - 1 / layer, y - 1, layer + 1)
    if node.right:
        graph.add_edge(node, node.right, weight='1')
        add_nodes_edges(node.right, graph, pos, x + 1 / layer, y - 1, layer + 1)
    return graph, pos

# heapq_tree 시각화
def draw_heapq_tree(root, title):
    plt.figure(figsize=(12, 6))
    G = nx.DiGraph()
    pos = {}
    G, pos = add_nodes_edges(root, G, pos)

    # 그래프 그리기
    labels = {node: node for node in G.nodes()}
    nx.draw(G, pos, labels=labels, with_labels=True, node_size=800, node_color='skyblue',
            font_weight='bold', font_size=8, font_color='darkred')
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red')
    plt.title(title)
    plt.show()


 # heapq_tree 시각화
draw_heapq_tree(root, title='Huffman Tree')

### [실습문제] 이미지 파일 허프만 코드로 압축

In [ ]:
from PIL import Image
import numpy as np

# 이미지 로딩 및 처리
img_path = "cat.jpg"
image = Image.open(img_path).convert("L")  # 흑백 변환
pixels = np.array(image).flatten().tolist()

# 허프만 압축 수행
root, code_map, encoded_output = huffman_tree(pixels)
original_bits = len(pixels) * 8

# 출력
print("\n✅ 입력 픽셀:", pixels)
print("\n✅ 허프만 코드맵:")
for char, code in code_map.items():
    print(f"  '{char}': {code}")

print("\n✅ 압축 결과:")
original_bits = len(pixels) * 8
compressed_bits = len(encoded_output)
compression_ratio = compressed_bits / original_bits

print(f"  - 원본 비트 수:     {original_bits:,} bits")
print(f"  - 압축 후 비트 수:  {compressed_bits:,} bits")
print(f"  - 압축 비율:        {compression_ratio:.2%}")
print(f"\nEncoded Output:\n{encoded_output}")


----------------

## 1-5.응용 : 그래프 색칠하기

### 1) 그래프 색칠 문제 (Graph Coloring Problem)
- 주어진 그래프에서 인접한 정점들이 서로 다른 색상으로 칠해지도록 하는 최소 색상의 수를 찾는 문제
- 그래프의 크기가 커질수록 최적해를 찾는 것이 매우 어려워
- 시간 복잡도:  $O(m^n)$, m은 가능한 색상 수, n은 정점의 수
- 웰치-포웰 알고리즘(Welch-Powell Algorithm)
- 특히 컴퓨터 과학, 통신 네트워크, 최적화 문제 등 여러 분야에서 응용됨
* **웰치-포웰 알고리즘의 기본 절차**
    - **정점의 정렬**: 모든 정점을 그 정점의 차수(연결된 간선의 수)에 따라 내림차순으로 정렬
    - **색 할당**: 가장 높은 차수를 가진 정점부터 시작하여, 사용 가능한 가장 낮은 번호의 색을 할당, 색은 이웃한 정점과 겹치지 않아야 함
    - **반복 처리**: 모든 정점이 색칠될 때까지 위 과정을 반복함

- **방법1**:  웰치-포웰 알고리즘 사용하여 그래프 색칠하기

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

def welch_powell(graph):
    # 정점의 차수에 따라 정렬
    degree_sequence = sorted(graph, key=lambda x: len(graph[x]), reverse=True)  #  차수 내림차순 정렬된 노드 리스트
    colors = {}  # (노드:색상)매핑된 정보 담는 딕셔너리
    color = 0    # 색상부여 시작 번호

    for node in degree_sequence:
        available_colors = set(range(color + 1))  # 사용 가능한 색상 집합 생성
        # 이미 색칠된 이웃 정점의 색상 제거
        available_colors.difference_update(colors.get(neighbor) for neighbor in graph[node] if neighbor in colors)
        if available_colors:
            colors[node] = min(available_colors)
        else:
            color += 1
            colors[node] = color

    return colors

# 그래프 시각화
def draw_grap(graph, coloring, pos=False):
    plt.figure(figsize=(5, 3))

    # 그래프 생성
    G = nx.Graph()
    for node in graph:
        G.add_node(node, color=f"C{coloring[node]}")

    for node, neighbors in graph.items():
        for i in neighbors:
            G.add_edge(node, i)

    # 그래프 시각화
    if not pos: pos = nx.spring_layout(G)
    colors = [G.nodes[node]['color'] for node in G.nodes]
    print(colors)  # 결과 출력: 각 노드의 색상 번호를 보여줌

    nx.draw(G, pos, node_color=colors, with_labels=True,  node_size=1000)
    plt.show()

In [ ]:
# 간단한 그래프 예제
graph = {
    'A': {'B', 'C'},
    'B': {'A', 'C', 'D', 'E'},
    'C': {'A', 'B', 'D'},
    'D': {'B', 'C', 'E'},
    'E': {'B', 'D'}
}
pos = {
    'A': (0, 0),
    'B': (1, 0),
    'C': (1,-1),
    'D': (2, 0),
    'E': (2,-1)
}

# 방법1 : welch_powell 알고리즘 사용
coloring = welch_powell(graph)
print(f"✅ 최소 색상 수: {len(set(coloring.values()))}")
print("\n✅ 정점별 색상: ")
for node in sorted(coloring):
    print(f" - Vertex {node} has color {coloring[node]}")

draw_grap(graph, coloring, pos)  # 그래프 시각화


- **방법2**: 그리디 알고리즘 적용: networkx greedy_color 함수 사용

In [ ]:
def draw_graph_greedy_color(graph, pos=False):
    plt.figure(figsize=(5, 3))

    # 그래프 생성
    G = nx.Graph()
    for node in graph:
        G.add_node(node)

    for node, neighbors in graph.items():
        for i in neighbors:
            G.add_edge(node, i)

    # 그래프 시각화
    if not pos: pos = nx.spring_layout(G)
    colors = nx.coloring.greedy_color(G, strategy='largest_first')
    # colors = [colors[node] for node in G.nodes()]
    colors = [f"C{colors[node]}" for node in G.nodes()]
    print(colors)  # 결과 출력: 각 노드의 색상 번호를 보여줌
    nx.draw(G, pos, node_color=colors, with_labels=True,  node_size=1000)
    plt.show()


# 함수 호출
draw_graph_greedy_color(graph, pos)


### [실습문제] 그래프 색칠하기  

In [ ]:
graph = {
    'A': {'B', 'C'},
    'B': {'A', 'D', 'E', 'F'},
    'C': {'A', 'D', 'I'},
    'D': {'B', 'C', 'E', 'G', 'I'},
    'E': {'B', 'D', 'F','G'},
    'F': {'B', 'E', 'H'},
    'G': {'D', 'E', 'H', 'I'},
    'H': {'F', 'G', 'I'},
    'I': {'C', 'D','G', 'H'}
}
pos = {
    'A': (0, 0),
    'B': (0,-2),
    'C': (2, 0),
    'D': (2,-1),
    'E': (1,-2),
    'F': (1,-3),
    'G': (3,-2),
    'H': (3,-3),
    'I': (4,-1)
}
coloring = welch_powell(graph)
print(f"✅ 최소 색상 수: {len(set(coloring.values()))}")
draw_grap(graph, coloring, pos)  # 그래프 시각화

In [ ]:
draw_graph_greedy_color(graph, pos)

### [실습문제] 과목별 보충시간 배정

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def draw_graph_coloring(G, coloring):

    # matplotlib의 색상 리스트 생성 (필요하면 더 추가 가능)
    colors_list = ['lightblue', 'lightgreen', 'salmon', 'gold', 'plum', 'coral', 'khaki', 'skyblue']

    # 각 노드의 색상 설정
    node_colors = [colors_list[coloring[node] % len(colors_list)] for node in G.nodes()]

    # 노드 위치 설정
    pos = nx.spring_layout(G, seed=42)

    # 그래프 그리기
    plt.figure(figsize=(8, 6))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1500)
    nx.draw_networkx_edges(G, pos)
    nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold')

    # 타이틀과 범례 설명
    plt.title("그래프 색칠 결과 (수업 시간표 교시 배정)")
    plt.axis('off')
    plt.show()


def graph_coloring(G):
    # 그리디 색칠 알고리즘을 사용하여 그래프 색칠
    # DSATUR(Degree of Saturation Algorithm)전략: 리디 기반 휴리스틱 알고리즘
    # 색 개수를 최소화하면서 인접한 정점들이 서로 다른 색을 갖도록 색칠하는 문제
    # largest_first: 점들을 차수가 높은 순서(degree 높은 순)로 정렬하여 그 순서대로 색칠
    coloring = nx.coloring.greedy_color(G, strategy='largest_first')
    return coloring


def create_graph():
    # 그래프 생성
    G = nx.Graph()

    # 각 수업을 수강하는 학생들
    classes = {
        '국어': ['A', 'C'],
        '영어': ['A', 'B'],
        '수학': ['A', 'D', 'E'],
        '과학': ['B', 'E'],
        '컴퓨터': ['C', 'D']
    }

    # 각 수업 쌍 간의 충돌을 그래프의 간선으로 추가
    for class1 in classes:
        for class2 in classes:
            if class1 != class2: # 같은 수업끼리는 비교하지 않음
                # 두 수업이 공통 학생을 가지면 간선을 추가
                if any(student in classes[class2] for student in classes[class1]):
                    G.add_edge(class1, class2)

    return G



def main():
    # 1.그래프 만들기
    G = create_graph()
    print("G edges: ", G.edges)

    # 2.그래프 색칠하기
    coloring = graph_coloring(G)

    # 3.고유한 색상 수 계산
    num_colors = len(set(coloring.values()))
    print(f"✅ 최소 색상 수: {num_colors}")

    print("\n✅ 각 수업의 색칠 결과(교시 배정):")
    for course, color in coloring.items():
        print(f" - {course}: {color + 1} 교시 ")  # 색상 번호를 교시 번호로 출력하기 위해 1을 더함

    # 그래프 시각화
    draw_graph_coloring(G, coloring)

main()
